In [2]:
import torch
import torch.nn as nn
torch.manual_seed(42)

$$
MSE = \frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2
$$

In [4]:
# 예측값과 정답값(회귀)
pred = torch.tensor([2.5, 5.0, 7.5])
target = torch.tensor([3.0, 5.0, 7.0])

# pytorch에서 제곱하는  MSELoss 함수
mse_buildin = nn.MSELoss()(pred, target)  # 객체를 함수처럼 호출 forward 정상적으로 값 태우는 것 

# 수식 기반
mse_menual = torch.mean((pred - target) ** 2)
print(f'mse_buildin : {mse_buildin}')
print(f'mse_menual : {mse_menual}')

mse_buildin : 0.1666666716337204
mse_menual : 0.1666666716337204


$$
BCE = -\frac{1}{n}\sum_{i=1}^{n}\left[y_i\log(\hat{y}_i)+(1-y_i)\log(1-\hat{y}_i)\right]
$$

In [6]:
# 단일 샘플 y = 1일때, 식은 -log(y_hat) 단순화되고.. 그래서 정답 확률이 높을수록 손실이 낮아지는 형태가 된다.
y = torch.tensor([1.0])
yhat_good = torch.tensor([0.9]) # 정답 확률이 높음
yhat_bad = torch.tensor([0.1]) # 정답 확률이 낮음
# 수작업 계산
bce_good_menual = -(y*torch.log(yhat_good) + (1-y)*torch.log(1-yhat_good))
bce_bad_menual = -(y*torch.log(yhat_bad) + (1-y)*torch.log(1-yhat_bad))

# 내장된 BCE
bce_good_buildin = nn.BCELoss()(yhat_good, y)
bce_bad_buildin = nn.BCELoss()(yhat_bad, y)

print(f'bce_good_menual : {bce_good_menual}')
print(f'bce_bad_menual : {bce_bad_menual}')
print(f'bce_good_buildin : {bce_good_buildin}')
print(f'bce_bad_buildin : {bce_bad_buildin}')

bce_good_menual : tensor([0.1054])
bce_bad_menual : tensor([2.3026])
bce_good_buildin : 0.10536054521799088
bce_bad_buildin : 2.3025851249694824


BCE 배치 계산   

In [8]:
prob_pred = torch.tensor([0.9,0.2,0.4])
label = torch.tensor([1.0,0.0,1.0])

nn.BCELoss()(prob_pred, label)

tensor(0.4149)

$$
CE = -\sum_{i=1}^{m} y_i\log(\hat{y}_i)
$$

In [26]:
# 클래스 3개 [고양이, 강아지, 토끼] 정답이 강아지 (index =1 )
target_index = torch.tensor([1])
one_hot = torch.tensor([[0.0, 1.0, 0.0 ]]) # 원 - 핫 인코딩된 정답

# 좋은 예측과 나쁜 예측
p_good = torch.tensor([0.2,0.7,0.1])
p_bad = torch.tensor([0.7, 0.2, 0.1])

# 수작업
ce_good_menual = -(one_hot*torch.log(p_good)).sum()
ce_bad_menual = -(one_hot*torch.log(p_bad)).sum()

# torch bce logits 입력 log(p) 수식에 사용된 log 값을 가짐
# torch.log(p_good).unsqueeze(0).unsqueeze(2).shape # .unsqueeze() tensor 에서 차수를 하나 더 만들어줌
ce_loss = nn.CrossEntropyLoss()
ce_good_builtin = ce_loss(torch.log(p_good).unsqueeze(0), target_index)
ce_bad_builtin = ce_loss(torch.log(p_bad).unsqueeze(0), target_index)

print(ce_good_menual, ce_good_builtin)
print(ce_bad_menual, ce_bad_builtin)

tensor(0.3567) tensor(0.3567)
tensor(1.6094) tensor(1.6094)


In [70]:
from sklearn.datasets import load_diabetes # 회귀
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn 
X,y = load_diabetes(return_X_y=True)

scaler = StandardScaler()
x_train,x_test,y_train, y_test = train_test_split(X,y, random_state=42)
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

x_train_t = torch.tensor(x_train, dtype=torch.float32)
x_test_t = torch.tensor(x_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

reg_model = nn.Sequential(
    nn.Linear(x_train_t.shape[1],64),
    nn.ReLU(),
    nn.Linear(64,1) # 이진 분류 1개, 다중분류일때만 클래스 개수
)

mse_loss = nn.MSELoss()
optimizer = torch.optim.Adam(reg_model.parameters(), lr=1e-2) # 0.01

epochs = 300
for _ in range(epochs):
    optimizer.zero_grad() # 이전 가중치를 초기화 해줌 , 중요 
    pred = reg_model(x_train_t) # 모델에 값을 넣어서 예측 
    loss = mse_loss(pred, y_train_t) # 오차 계산
    loss.backward() # 가중치 찾아서 각 계산과정에 배치 .backward()
    optimizer.step() # 각 계산과정의 가중치와 바이어스를 업데이트

with torch.no_grad(): # 가중치를 사용하지 않는다. 
    test_pred = reg_model(x_test_t)
    test_mse = mse_loss(test_pred, y_test_t)

# test_mse 텐서에 들어 있음 출력이나 기타 산술연산이 필요함 꺼내야함.. item()

print(f'test mse : {test_mse.item():.4f}')

from sklearn.metrics import r2_score
y_true = y_test_t.tolist()
y_pred = test_pred.tolist()

r2_score(y_true, y_pred)

test mse : 2961.7102


0.46439850897505475

In [79]:
import torch
import torch.nn as nn
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. 딥러닝 레이어 sequential에서 입력을 받는 부분을  수정 피처개수
# 2 출력레이어의 활성화 함수로 시그모이드
# 3. 손실함수 nn.BCELoss()
# 추론  with .....  0.5보다 크면 1 그렇지 않으면 0 변경

X, y = load_breast_cancer(return_X_y=True)

scaler = StandardScaler()
x_train, x_test, y_train, y_test = train_test_split(X, y, random_state=42)
x_train = scaler.fit_transform(x_train)
x_test  = scaler.transform(x_test)

x_train_t = torch.tensor(x_train, dtype=torch.float32)
x_test_t  = torch.tensor(x_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  
y_test_t  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)  


model = nn.Sequential(
    nn.Linear(x_train_t.shape[1], 64),
    nn.ReLU(),
    nn.Linear(64, 1),
    nn.Sigmoid()
)


bce_loss  = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


for i in range(300):
    optimizer.zero_grad()
    pred_model = model(x_train_t)
    loss = bce_loss(pred_model, y_train_t)
    loss.backward()
    optimizer.step()        


with torch.no_grad():
    pred_prob  = model(x_test_t) 
    test_bce = bce_loss(pred_prob, y_test_t)                   
    pred_label = (pred_prob >= 0.5).float()           

    accuracy = (pred_label == y_test_t).float().mean()
    print(accuracy)

tensor(0.9860)


In [ ]:
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
# 데이터 변경
# 신경망의 입력 데이터 개수
# 신경망의 최종 출력의 뉴런 수 3
# 최종출력의 마지막에 활성함수 softmax를 사용 안함 why?? 손실함수에서 확률 분포로 변경하는 기능 내장 <- 케라스만 

# 손실 함수 nn.CrossEntropyLoss()

# epochs 학습하는 부분까지 완성

# 추론 -->
X, y = load_iris(return_X_y=True)

scaler = StandardScaler()
x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=42)
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

x_train_t = torch.tensor(x_train,dtype=torch.float32)
x_test_t = torch.tensor(x_test,dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)


model = nn.Sequential(
    nn.Linear(x_train_t.shape[1],64),
    nn.ReLU(),
    nn.Linear(64,3),
    nn.Softmax()
)

ce_loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 300
for i in range(epochs):
    optimizer.zero_grad()
    logits = model(x_train_t)
    loss = ce_loss(logits, y_train_t)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    test_pred = model(x_test_t)
    test_mse = ce_loss(test_pred, y_test_t)

(torch.argmax(test_pred, dim=1) == y_test_t).float().mean()

c:\Users\Playdata\miniconda3\envs\test_env\lib\site-packages\torch\nn\modules\module.py:1511: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


tensor(1.)

y = wx
#
x,y = 2,4
#
w = 1
#
lr = 0.1
#
y_hat = 1 x 2 = 2
#
손실 L = (4-2)**2 = 4
#
기울기 = -(2x2x2) = -8

w = 1.8
yhat = 1.8 * 2 = 3.6
손실 L = (4-3,6)**2 = 0.16

dl/dw = -2*2(y-1.8*2) = 1.6
w = 1.8 - 0.1* 1.6 = 